In [1]:
!git clone  https://github.com/deepinsight/insightface

fatal: destination path 'insightface' already exists and is not an empty directory.


In [2]:
# Fix the requirements.txt file
###%cd /kaggle/working/insightface/recognition/arcface_torch
#%cd /Atten dualFace Project/insightface/recognition/arcface_torch
#%cd Atten dualFace Project /insightface/recognition/arcface_torch
#%cd "insightface/recognition/arcface_torch"
%cd "/slurm/homes/bel/Atten dualFace Project /insightface/recognition/arcface_torch"
# Read and fix the requirements
with open('requirement.txt', 'r') as f:
    content = f.read()

# Replace 'sklearn' with 'scikit-learn'
content = content.replace('sklearn', 'scikit-learn')

with open('requirement.txt', 'w') as f:
    f.write(content)

# Now install
!pip install -r requirement.txt
! pip install easydict mxnet tensorboard insightface


/slurm/homes/bel/Atten dualFace Project /insightface/recognition/arcface_torch
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [3]:
# Create custom config file
config_content = '''
from easydict import EasyDict as edict

config = edict()
config.margin_list = (.0, 0.5, 0.0)
config.network = "r100"  # Options: r18, r34, r50, r100, mbf (MobileFaceNet)
config.resume = False
config.output = "Atten dualFace Project /output"
config.embedding_size = 512
config.sample_rate = .0
config.fp16 = True
config.momentum = 0.9
config.weight_decay = 5e-4
config.batch_size = 256  # Reduced for Kaggle GPU memory
config.lr = 0.1
config.verbose = 2500
config.dali = False
config.seed = 2048
config.num_workers = 8
config.gradient_acc = 1

# Dataset paths - UPDATE THESE to your Kaggle dataset paths
config.rec = "/slurm/homes/bel/Atten dualFace Project /faces_emore/faces_emore"  # Path to your MS1MV2 dataset
config.num_classes = 85742
config.num_image = 5822653
config.num_epoch = 25
config.warmup_epoch = 1
config.val_targets = ["lfw", "cfp_fp", "agedb_30", "calfw","cfp_ff","cplfw"]

# Additional settings
config.optimizer = "sgd"
config.frequent = 100
config.interclass_filtering_threshold = 0
config.save_all_states = True

# WandB (optional - set to False if not using)
config.using_wandb = False
config.wandb_key = ""
config.wandb_entity = ""
config.wandb_project = ""
config.wandb_resume = False
config.suffix_run_name = None
config.notes = ""
config.wandb_log_all = False
config.save_artifacts = False
'''

# Write the config file
import os
os.makedirs('insightface/recognition/arcface_torch/configs', exist_ok=True)
with open('insightface/recognition/arcface_torch/configs/ms1mv2_kaggle.py', 'w') as f:
    f.write(config_content)

print("Config file created!")

Config file created!


In [4]:
#!pip install numpy==1.23.5


In [5]:
import torch

# import torch_xla
# print(f"PyTorch XLA version: {torch_xla.__version__}")

# import torch_xla.core.xla_model as xm
# print("Available functions:", [x for x in dir(xm) if not x.startswith('_')])
# torch_xla.device()

# USE THIS INSTEAD:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device selected: {device}")

Device selected: cuda


In [6]:
#%cd "insightface/recognition/arcface_torch"
%cd "/slurm/homes/bel/Atten dualFace Project /insightface/recognition/arcface_torch"

# Create a modified train script that doesn't use TensorBoard
train_code = '''
import argparse
import logging
import os
from datetime import datetime

import numpy as np
import torch
from backbones import get_model
from dataset import get_dataloader
from losses import CombinedMarginLoss
from lr_scheduler import PolynomialLRWarmup
from partial_fc_v2 import PartialFC_V2
from torch import distributed
from torch.utils.data import DataLoader
from utils.utils_config import get_config
from utils.utils_distributed_sampler import setup_seed
from utils.utils_logging import AverageMeter, init_logging
from torch.distributed.algorithms.ddp_comm_hooks.default_hooks import fp16_compress_hook

assert torch.__version__ >= ".12.0", "In order to enjoy the features of the new torch, we have upgraded the torch to .12.0."

try:
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    distributed.init_process_group("nccl")
except KeyError: 
    rank = 0
    local_rank = 0
    world_size = 1
    distributed.init_process_group(
        backend="nccl",
        init_method="tcp://127.0.0.1:12584",
        rank=rank,
        world_size=world_size,
    )


def main(args):
    cfg = get_config(args.config)
    setup_seed(seed=cfg.seed, cuda_deterministic=False)

    torch.cuda.set_device(local_rank)
    os.makedirs(cfg.output, exist_ok=True)
    init_logging(rank, cfg.output)

    train_loader = get_dataloader(
        cfg.rec,
        local_rank,
        cfg.batch_size,
        cfg.dali,
        cfg.dali_aug,
        cfg.seed,
        cfg.num_workers
    )

    backbone = get_model(
        cfg.network, dropout=0.0, fp16=cfg.fp16, num_features=cfg.embedding_size).cuda()

    backbone = torch.nn.parallel.DistributedDataParallel(
        module=backbone, broadcast_buffers=False, device_ids=[local_rank], bucket_cap_mb=16,
        find_unused_parameters=True)
    backbone.register_comm_hook(None, fp16_compress_hook)

    backbone.train()
    backbone._set_static_graph()

    margin_loss = CombinedMarginLoss(
        64,
        cfg.margin_list[0],
        cfg.margin_list[1],
        cfg.margin_list[2],
        cfg.interclass_filtering_threshold
    )

    if cfg.optimizer == "sgd":
        module_partial_fc = PartialFC_V2(
            margin_loss, cfg.embedding_size, cfg.num_classes,
            cfg.sample_rate, False)
        module_partial_fc.train().cuda()
        opt = torch.optim.SGD(
            params=[{"params": backbone.parameters()}, {"params": module_partial_fc.parameters()}],
            lr=cfg.lr, momentum=0.9, weight_decay=cfg.weight_decay)

    elif cfg.optimizer == "adamw":
        module_partial_fc = PartialFC_V2(
            margin_loss, cfg.embedding_size, cfg.num_classes,
            cfg.sample_rate, False)
        module_partial_fc.train().cuda()
        opt = torch.optim.AdamW(
            params=[{"params": backbone.parameters()}, {"params": module_partial_fc.parameters()}],
            lr=cfg.lr, weight_decay=cfg.weight_decay)
    else:
        raise ValueError(f"Unknown optimizer: {cfg.optimizer}")

    cfg.total_batch_size = cfg.batch_size * world_size
    cfg.warmup_step = cfg.num_image // cfg.total_batch_size * cfg.warmup_epoch
    cfg.total_step = cfg.num_image // cfg.total_batch_size * cfg.num_epoch

    lr_scheduler = PolynomialLRWarmup(
        optimizer=opt,
        warmup_iters=cfg.warmup_step,
        total_iters=cfg.total_step)

    start_epoch = 0
    global_step = 0
    if cfg.resume:
        dict_checkpoint = torch.load(os.path.join(cfg.output, f"checkpoint_gpu_{rank}.pt"))
        start_epoch = dict_checkpoint["epoch"]
        global_step = dict_checkpoint["global_step"]
        backbone.module.load_state_dict(dict_checkpoint["state_dict_backbone"])
        module_partial_fc.load_state_dict(dict_checkpoint["state_dict_softmax_fc"])
        opt.load_state_dict(dict_checkpoint["state_optimizer"])
        lr_scheduler.load_state_dict(dict_checkpoint["state_lr_scheduler"])
        del dict_checkpoint

    for key, value in cfg.items():
        num_space = 25 - len(key)
        logging.info(":  " + key + " " * num_space + str(value))

    loss_am = AverageMeter()
    amp = torch.cuda.amp.grad_scaler.GradScaler(growth_interval=100)

    for epoch in range(start_epoch, cfg.num_epoch):
        if isinstance(train_loader, DataLoader):
            train_loader.sampler.set_epoch(epoch)
        
        for batch_idx, (img, local_labels) in enumerate(train_loader):
            global_step += 1
            local_embeddings = backbone(img)
            loss:  torch.Tensor = module_partial_fc(local_embeddings, local_labels)

            if cfg.fp16:
                amp.scale(loss).backward()
                if global_step % cfg.gradient_acc == 0:
                    amp.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(backbone.parameters(), 5)
                    amp.step(opt)
                    amp.update()
                    opt.zero_grad()
            else:
                loss.backward()
                if global_step % cfg.gradient_acc == 0:
                    torch.nn.utils.clip_grad_norm_(backbone.parameters(), 5)
                    opt.step()
                    opt.zero_grad()
            lr_scheduler.step()

            with torch.no_grad():
                loss_am.update(loss.item(), 1)
                if global_step % cfg.frequent == 0:
                    lr = lr_scheduler.get_last_lr()[0]
                    logging.info(
                        f"Epoch:  {epoch}, Step: {global_step}, Loss: {loss_am.avg:.4f}, LR: {lr:.6f}"
                    )

        # Save checkpoint after each epoch
        if cfg.save_all_states: 
            checkpoint = {
                "epoch": epoch + 1,
                "global_step":  global_step,
                "state_dict_backbone": backbone.module.state_dict(),
                "state_dict_softmax_fc": module_partial_fc.state_dict(),
                "state_optimizer":  opt.state_dict(),
                "state_lr_scheduler":  lr_scheduler.state_dict()
            }
            torch.save(checkpoint, os.path.join(cfg.output, f"checkpoint_gpu_{rank}.pt"))

        if rank == 0:
            path_module = os.path.join(cfg.output, "model.pt")
            torch.save(backbone.module.state_dict(), path_module)
            logging.info(f"Model saved to {path_module}")

        if cfg.dali:
            train_loader.reset()

    if rank == 0:
        path_module = os.path.join(cfg.output, "model.pt")
        torch.save(backbone.module.state_dict(), path_module)
        logging.info(f"Final model saved to {path_module}")


if __name__ == "__main__": 
    torch.backends.cudnn.benchmark = True
    parser = argparse.ArgumentParser(description="Distributed Arcface Training in Pytorch")
    parser.add_argument("config", type=str, help="py config file")
    main(parser.parse_args())
'''

with open('train_kaggle.py', 'w') as f:
    f.write(train_code)

print("✓ Created train_kaggle.py (TensorBoard-free version)")

/slurm/homes/bel/Atten dualFace Project /insightface/recognition/arcface_torch
✓ Created train_kaggle.py (TensorBoard-free version)


In [7]:
# Check if you have the required files
import os

# Update this path to your actual dataset path
dataset_path = "/slurm/homes/bel/Atten dualFace Project /faces_emore/faces_emore"  # Change this!

# List all files
for f in os.listdir(dataset_path):
    filepath = os.path.join(dataset_path, f)
    size = os.path.getsize(filepath) / (1024**3)  # Size in GB
    print(f"{f}:  {size:.2f} GB" if size > 0.01 else f"{f}: {os.path.getsize(filepath)/1024:.2f} KB")

cfp_fp.bin:  0.07 GB
property: 0.01 KB
vgg2_fp.bin:  0.06 GB
cplfw.bin:  0.06 GB
train.idx:  0.11 GB
cfp_ff.bin:  0.08 GB
agedb_30.bin:  0.07 GB
calfw.bin:  0.07 GB
train.rec:  15.40 GB
lfw.bin:  0.06 GB


In [8]:
#%cd "insightface/recognition/arcface_torch"
%cd "/slurm/homes/bel/Atten dualFace Project /insightface/recognition/arcface_torch"

# Create the attention modules file
attention_code = '''
"""
Custom Attention Modules for Ablation Studies
==============================================
Supports the following configurations:
- baseline: No attention (standard ArcFace)
- dual_attention: Original AttentionBlock
- uncertainty: UncertaintyAttention only
- channel:  ChannelAttentionBlock only  
- hybrid: HybridAttentionBlock (Channel + Uncertainty)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class AttentionBlock(nn.Module):
    """
    Baseline Dual Attention Block
    Input: 512 embedding
    Output:  Scalar score between 0.5 and .5
    """
    def __init__(self, in_features):
        super(AttentionBlock, self).__init__()
        self.fc1 = nn.Linear(in_features, 128)
        self.relu = nn.PReLU()
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        raw_score = self.sigmoid(x)
        attention_score = 0.5 + raw_score
        return attention_score


class ChannelAttentionBlock(nn.Module):
    """
    Channel Attention using Squeeze-and-Excitation
    """
    def __init__(self, in_features, reduction=64):
        super(ChannelAttentionBlock, self).__init__()
        self.se = nn.Sequential(
            nn.Linear(in_features, in_features // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_features // reduction, in_features, bias=False),
            nn.Sigmoid()
        )
        self.final_score = nn.Linear(in_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        importance = self.se(x)
        x_refined = x * importance
        out = self.final_score(x_refined)
        raw_score = self.sigmoid(out)
        return 0.5 + raw_score


class UncertaintyAttention(nn.Module):
    """
    Uncertainty-based Attention using Log-Variance prediction
    """
    def __init__(self, in_features):
        super(UncertaintyAttention, self).__init__()
        self.fc_log_var = nn.Linear(in_features, 1)
        self.bn = nn.BatchNorm1d(1)

    def forward(self, x):
        log_var = self.fc_log_var(x)
        log_var = self.bn(log_var)
        sigma = torch.exp(0.5 * log_var)
        atten_score = .0 / (sigma + 0.1)
        atten_score = torch.clamp(atten_score, 0.5, .5)
        return atten_score


class HybridAttentionBlock(nn.Module):
    """
    Hybrid Attention:  Channel Attention + Uncertainty Estimation
    Step 1: Channel Attention (Filter noise from features)
    Step 2: Uncertainty Estimation (Calculate confidence based on clean features)
    """
    def __init__(self, in_features, reduction=64):
        super(HybridAttentionBlock, self).__init__()
        
        # Channel Attention (The Filter)
        self.channel_fc1 = nn.Linear(in_features, in_features // reduction, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.channel_fc2 = nn.Linear(in_features // reduction, in_features, bias=False)
        self.sigmoid = nn.Sigmoid()

        # Uncertainty Estimation (The Judge)
        self.uncertainty_fc = nn.Linear(in_features, 1)
        self.uncertainty_bn = nn.BatchNorm1d(1)

    def forward(self, x):
        # .Channel Attention - Clean the features
        weights = self.channel_fc1(x)
        weights = self.relu(weights)
        weights = self.channel_fc2(weights)
        weights = self.sigmoid(weights)
        x_refined = x * weights
        
        # 2.Uncertainty Estimation
        log_var = self.uncertainty_fc(x_refined)
        log_var = self.uncertainty_bn(log_var)
        sigma = torch.exp(0.5 * log_var)
        
        # 3.Convert to attention score [0.5, .5]
        raw_score = .0 / (.0 + sigma)
        attention_score = 0.5 + raw_score
        
        return attention_score


class NoAttention(nn.Module):
    """
    Baseline: No attention, returns constant score of .0
    """
    def __init__(self, in_features):
        super(NoAttention, self).__init__()
        self.in_features = in_features

    def forward(self, x):
        batch_size = x.size(0)
        return torch.ones(batch_size, 1, device=x.device)


def get_attention_module(attention_type, in_features, reduction=32):
    """
    Factory function to get attention module based on type
    
    Args:
        attention_type: One of ['baseline', 'dual_attention', 'uncertainty', 'channel', 'hybrid']
        in_features: Input feature dimension (usually 512)
        reduction: Reduction ratio for channel attention
    
    Returns: 
        Attention module
    """
    attention_modules = {
        'baseline': NoAttention,
        'dual_attention':  AttentionBlock,
        'uncertainty':  UncertaintyAttention,
        'channel': ChannelAttentionBlock,
        'hybrid':  HybridAttentionBlock,
    }
    
    if attention_type not in attention_modules:
        raise ValueError(f"Unknown attention type: {attention_type}."
                        f"Choose from {list(attention_modules.keys())}")
    
    module_class = attention_modules[attention_type]
    
    if attention_type in ['channel', 'hybrid']: 
        return module_class(in_features, reduction=reduction)
    else: 
        return module_class(in_features)
'''

with open('attention_modules.py', 'w') as f:
    f.write(attention_code)

print("✓ Created attention_modules.py")



/slurm/homes/bel/Atten dualFace Project /insightface/recognition/arcface_torch
✓ Created attention_modules.py


In [9]:
# Create the custom loss with attention
loss_code = '''
"""
Attention-based Dual Margin Loss
================================
Integrates with InsightFace's PartialFC framework
Supports ablation studies with different attention mechanisms
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
import math

from attention_modules import get_attention_module


class AttenDualMarginLoss(nn.Module):
    """
    Attention-based Dual Margin Loss
    
    Combines:
    - Adaptive positive margin (based on attention score)
    - Fixed negative margin
    - Support for different attention mechanisms
    """
    def __init__(
        self,
        scale=64.0,
        m1=0.5,           # Base positive margin
        m2=0.1,           # Negative margin
        attention_type='hybrid',  # baseline, dual_attention, uncertainty, channel, hybrid
        embedding_size=512,
        reduction=32,
    ):
        super(AttenDualMarginLoss, self).__init__()
        self.scale = scale
        self.m1 = m1
        self.m2 = m2
        self.attention_type = attention_type
        
        # Get attention module
        self.attention_net = get_attention_module(
            attention_type, embedding_size, reduction
        )
        
        print(f"[AttenDualMarginLoss] Attention Type: {attention_type}")
        print(f"[AttenDualMarginLoss] Scale: {scale}, M1: {m1}, M2: {m2}")

    def forward(self, logits, labels):
        """
        Args: 
            logits:  Cosine similarity scores (B, num_classes)
            labels: Ground truth labels (B,)
        
        Returns:
            Modified logits with dual margin applied
        """
        # This will be called from PartialFC_V2 with cosine similarities
        # We need to store embeddings separately for attention calculation
        raise NotImplementedError("Use AttenDualPartialFC instead")


class AttenDualPartialFC(nn.Module):
    """
    Partial FC with Attention-based Dual Margin
    Drop-in replacement for PartialFC_V2
    """
    def __init__(
        self,
        embedding_size=512,
        num_classes=85742,
        sample_rate=.0,
        scale=64.0,
        m1=0.5,
        m2=0.1,
        attention_type='hybrid',
        reduction=32,
    ):
        super(AttenDualPartialFC, self).__init__()
        
        self.embedding_size = embedding_size
        self.num_classes = num_classes
        self.sample_rate = sample_rate
        self.scale = scale
        self.m1 = m1
        self.m2 = m2
        self.attention_type = attention_type
        
        # Class weights
        self.weight = Parameter(torch.FloatTensor(num_classes, embedding_size))
        nn.init.xavier_uniform_(self.weight)
        
        # Attention module
        self.attention_net = get_attention_module(
            attention_type, embedding_size, reduction
        )
        
        # For logging
        self.last_attention_score = None
        
        print(f"[AttenDualPartialFC] Initialized")
        print(f"  - Attention:  {attention_type}")
        print(f"  - Classes: {num_classes}")
        print(f"  - Scale: {scale}, M1: {m1}, M2: {m2}")

    def forward(self, embeddings, labels):
        """
        Args:
            embeddings: Feature embeddings (B, embedding_size)
            labels: Ground truth labels (B,)
        
        Returns:
            loss: Cross entropy loss
        """
        # .Normalize embeddings and weights
        embeddings_norm = F.normalize(embeddings, dim=1)
        weight_norm = F.normalize(self.weight, dim=1)
        
        # 2.Compute cosine similarity
        cosine = F.linear(embeddings_norm, weight_norm)
        
        # 3.Get attention scores
        atten_score = self.attention_net(embeddings)  # (B, 1)
        self.last_attention_score = atten_score.mean().item()
        
        # 4.Clamp cosine for numerical stability
        cosine = torch.clamp(cosine, -.0 + 1e-7, .0 - 1e-7)
        
        # 5.Convert to angles
        theta = torch.acos(cosine)
        
        # 6.Calculate adaptive positive margin
        m1_dynamic = self.m1 * atten_score  # (B, 1)
        
        # 7.Create one-hot mask
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), .0)
        
        # 8.Apply dual margins
        # Target class: add positive margin
        target_angle = theta + m1_dynamic
        target_cosine = torch.cos(target_angle)
        
        # Non-target classes: subtract negative margin
        non_target_angle = theta - self.m2
        non_target_cosine = torch.cos(non_target_angle)
        
        # 9.Combine
        output = one_hot * target_cosine + (.0 - one_hot) * non_target_cosine
        
        # 10.Scale
        output = output * self.scale
        
        # 1.Compute loss
        loss = F.cross_entropy(output, labels)
        
        return loss

    def get_attention_score(self):
        """Return last computed attention score for logging"""
        return self.last_attention_score if self.last_attention_score else 0.0
'''

with open('losses_attention.py', 'w') as f:
    f.write(loss_code)

print("✓ Created losses_attention.py")

✓ Created losses_attention.py


In [10]:
import os

# 1. Create the directory first (This fixes your current error)
os.makedirs('configs', exist_ok=True)

# 2. Define the config content (Updated with correct paths)
ablation_config = '''
"""
Ablation Study Configuration
============================
Run different attention mechanisms for comparison
"""

from easydict import EasyDict as edict

def get_ablation_config(attention_type="hybrid"):
    """
    Get config for ablation study
    Args:
        attention_type: One of: 
            - 'baseline': No attention (standard margins)
            - 'dual_attention':  Original simple attention
            - 'uncertainty': Uncertainty-based attention
            - 'channel':  Channel attention (SE-style)
            - 'hybrid': Channel + Uncertainty (full model)
    """
    config = edict()
    
    # ============== Model Settings ==============
    config.network = "r100"
    config.embedding_size = 512
    config.fp16 = True
    
    # ============== Attention Settings ==============
    config.attention_type = attention_type
    config.attention_reduction = 64  # For channel/hybrid attention
    
    # ============== Loss Settings ==============
    config.scale = 64.0
    config.m1 = 0.5          # Base positive margin
    config.m2 = 0.1          # Negative margin
    
    # ============== Training Settings ==============
    config.optimizer = "sgd"
    config.lr = 0.1
    config.batch_size = 256
    config.num_epoch = 25
    config.warmup_epoch = 1
    config.weight_decay = 5e-4
    config.momentum = 0.9
    config.gradient_acc = 1
    
    # ============== Dataset Settings ==============
    # UPDATED PATH: Go up 3 levels to find the data
    config.rec = "/slurm/homes/bel/Atten dualFace Project /faces_emore/faces_emore" 
    config.num_classes = 85742
    config.num_image = 5822653
    
    # ============== Validation Settings ==============
    config.val_targets = ["lfw", "cfp_fp", "agedb_30", "calfw","cfp_ff","cplfw"]
    
    # ============== Output Settings ==============
    # UPDATED PATH: Go up 3 levels to find the output folder
    config.output = f"output/output_{attention_type}"
    config.save_all_states = True
    config.resume = False
    config.verbose = 2500
    config.frequent = 100
    
    # ============== Other Settings ==============
    config.seed = 2048
    config.num_workers = 8
    config.dali = False
    config.dali_aug = False
    config.sample_rate = 1.0
    config.using_wandb = False
    
    return config


# Pre-defined configs for each ablation
ABLATION_CONFIGS = {
    'baseline': get_ablation_config('baseline'),
    'dual_attention': get_ablation_config('dual_attention'),
    'uncertainty':  get_ablation_config('uncertainty'),
    'channel': get_ablation_config('channel'),
    'hybrid': get_ablation_config('hybrid'),
}
'''

# 3. Write the file
with open('configs/ablation_config.py', 'w') as f:
    f.write(ablation_config)

print("✓ Created configs/ablation_config.py")

✓ Created configs/ablation_config.py


In [11]:
%cd "/slurm/homes/bel/Atten dualFace Project /insightface/recognition/arcface_torch"

# Create a fully self-contained training script with NO TensorBoard dependencies
ablation_train = '''
"""
Ablation Study Training Script (Self-Contained)
================================================
No TensorBoard/TensorFlow dependencies

BASELINE:  dual_attention (original AttentionBlock)
ABLATIONS:
  - uncertainty:  UncertaintyAttention
  - channel:  ChannelAttentionBlock  
  - hybrid: HybridAttentionBlock (Full Model)
"""

import argparse
import logging
import os
import sys
import json
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
from torch import distributed
from torch.utils.data import DataLoader
from torch.distributed.algorithms.ddp_comm_hooks.default_hooks import fp16_compress_hook
from easydict import EasyDict as edict

# ============================================================
# ATTENTION MODULES
# ============================================================

class DualAttention(nn.Module):
    """BASELINE: Original simple MLP attention"""
    def __init__(self, in_features, **kwargs):
        super(DualAttention, self).__init__()
        self.fc1 = nn.Linear(in_features, 128)
        self.relu = nn.PReLU()
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        raw_score = self.sigmoid(x)
        return 0.5 + raw_score


class UncertaintyAttention(nn.Module):
    """ABLATION: Log-variance uncertainty estimation"""
    def __init__(self, in_features, **kwargs):
        super(UncertaintyAttention, self).__init__()
        self.fc_log_var = nn.Linear(in_features, 1)
        self.bn = nn.BatchNorm1d(1)

    def forward(self, x):
        log_var = self.fc_log_var(x)
        log_var = self.bn(log_var)
        sigma = torch.exp(0.5 * log_var)
        atten_score = 1.0 / (sigma + 0.1)
        atten_score = torch.clamp(atten_score, 0.5, 1.5)
        return atten_score


class ChannelAttention(nn.Module):
    """ABLATION:  Squeeze-Excitation style attention"""
    def __init__(self, in_features, reduction=32, **kwargs):
        super(ChannelAttention, self).__init__()
        self.se = nn.Sequential(
            nn.Linear(in_features, in_features // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_features // reduction, in_features, bias=False),
            nn.Sigmoid()
        )
        self.final_score = nn.Linear(in_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        importance = self.se(x)
        x_refined = x * importance
        out = self.final_score(x_refined)
        raw_score = self.sigmoid(out)
        return 0.5 + raw_score


class HybridAttention(nn.Module):
    """FULL MODEL: Channel + Uncertainty"""
    def __init__(self, in_features, reduction=64, **kwargs):
        super(HybridAttention, self).__init__()
        self.channel_fc1 = nn.Linear(in_features, in_features // reduction, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.channel_fc2 = nn.Linear(in_features // reduction, in_features, bias=False)
        self.sigmoid = nn.Sigmoid()
        self.uncertainty_fc = nn.Linear(in_features, 1)
        self.uncertainty_bn = nn.BatchNorm1d(1)

    def forward(self, x):
        weights = self.channel_fc1(x)
        weights = self.relu(weights)
        weights = self.channel_fc2(weights)
        weights = self.sigmoid(weights)
        x_refined = x * weights
        
        log_var = self.uncertainty_fc(x_refined)
        log_var = self.uncertainty_bn(log_var)
        sigma = torch.exp(0.5 * log_var)
        
        raw_score = 1.0 / (1.0 + sigma)
        return 0.5 + raw_score


ATTENTION_REGISTRY = {
    'dual_attention': DualAttention,
    'uncertainty':  UncertaintyAttention,
    'channel': ChannelAttention,
    'hybrid': HybridAttention,
}

def get_attention_module(attention_type, in_features, reduction=32):
    if attention_type not in ATTENTION_REGISTRY:
        raise ValueError(f"Unknown:  {attention_type}.Available: {list(ATTENTION_REGISTRY.keys())}")
    return ATTENTION_REGISTRY[attention_type](in_features, reduction=reduction)


# ============================================================
# LOSS MODULE
# ============================================================

class AttenDualPartialFC(nn.Module):
    """Attention-based Dual Margin Loss"""
    def __init__(self, embedding_size, num_classes, scale=64.0, m1=0.5, m2=0.1,
                 attention_type='dual_attention', reduction=32, sample_rate=1.0):
        super(AttenDualPartialFC, self).__init__()
        
        self.embedding_size = embedding_size
        self.num_classes = num_classes
        self.scale = scale
        self.m1 = m1
        self.m2 = m2
        self.attention_type = attention_type
        
        self.weight = Parameter(torch.FloatTensor(num_classes, embedding_size))
        nn.init.xavier_uniform_(self.weight)
        
        self.attention_net = get_attention_module(attention_type, embedding_size, reduction)
        self.last_attention_score = None
        
        print(f"[AttenDualPartialFC] {attention_type} | Classes: {num_classes} | s={scale}, m1={m1}, m2={m2}")

    def forward(self, embeddings, labels):
        embeddings_norm = F.normalize(embeddings, dim=1)
        weight_norm = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings_norm, weight_norm)
        
        atten_score = self.attention_net(embeddings)
        self.last_attention_score = atten_score.mean().item()
        
        cosine = torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7)
        theta = torch.acos(cosine)
        
        m1_dynamic = self.m1 * atten_score
        
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        
        target_angle = theta + m1_dynamic
        target_cosine = torch.cos(target_angle)
        
        non_target_angle = theta - self.m2
        non_target_cosine = torch.cos(non_target_angle)
        
        output = one_hot * target_cosine + (1.0 - one_hot) * non_target_cosine
        output = output * self.scale
        
        return F.cross_entropy(output, labels)

    def get_attention_score(self):
        return self.last_attention_score if self.last_attention_score else 0.0


# ============================================================
# SIMPLE VALIDATION (No TensorBoard)
# ============================================================

import pickle
import sklearn
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA

def load_bin(path, image_size=(112, 112)):
    """Load validation bin file"""
    try:
        import mxnet as mx
        with open(path, 'rb') as f:
            bins, issame_list = pickle.load(f, encoding='bytes')
    except:
        with open(path, 'rb') as f:
            bins, issame_list = pickle.load(f, encoding='bytes')
    
    data_list = []
    for flip in [0, 1]: 
        data = torch.empty((len(issame_list) * 2, 3, image_size[0], image_size[1]))
        data_list.append(data)
    
    for idx in range(len(issame_list) * 2):
        _bin = bins[idx]
        try:
            import mxnet as mx
            img = mx.image.imdecode(_bin).asnumpy()
        except:
            import cv2
            img = cv2.imdecode(np.frombuffer(_bin, np.uint8), cv2.IMREAD_COLOR)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        img = torch.from_numpy(img.transpose(2, 0, 1)).float()
        img = (img / 255.0 - 0.5) / 0.5
        
        for flip in [0, 1]:
            if flip == 1:
                img = torch.flip(img, [2])
            data_list[flip][idx] = img
    
    return data_list, issame_list


def evaluate_accuracy(embeddings, issame_list, nrof_folds=10):
    """Calculate accuracy using k-fold"""
    embeddings1 = embeddings[0:: 2]
    embeddings2 = embeddings[1::2]
    
    diff = embeddings1 - embeddings2
    dist = np.sum(np.square(diff), 1)
    
    thresholds = np.arange(0, 4, 0.01)
    issame = np.array(issame_list)
    
    accuracy_list = []
    for threshold in thresholds:
        predict_issame = np.less(dist, threshold)
        accuracy = np.mean(predict_issame == issame)
        accuracy_list.append(accuracy)
    
    best_acc = max(accuracy_list)
    best_threshold = thresholds[np.argmax(accuracy_list)]
    
    return best_acc, best_threshold


@torch.no_grad()
def validate(backbone, data_path, val_targets, batch_size=64):
    """Run validation on targets"""
    backbone.eval()
    results = {}
    
    for name in val_targets: 
        bin_path = os.path.join(data_path, f"{name}.bin")
        if not os.path.exists(bin_path):
            print(f"  [Val] {name}:  NOT FOUND")
            continue
        
        try:
            data_list, issame_list = load_bin(bin_path)
            
            embeddings_list = []
            for data in data_list: 
                embeddings = []
                for i in range(0, len(data), batch_size):
                    batch = data[i:i+batch_size].cuda()
                    emb = backbone(batch).cpu().numpy()
                    embeddings.append(emb)
                embeddings_list.append(np.concatenate(embeddings))
            
            embeddings = embeddings_list[0] + embeddings_list[1]
            embeddings = sklearn.preprocessing.normalize(embeddings)
            
            acc, threshold = evaluate_accuracy(embeddings, issame_list)
            results[name] = acc
            print(f"  [Val] {name}: {acc*100:.2f}%")
        except Exception as e: 
            print(f"  [Val] {name}: ERROR - {e}")
    
    backbone.train()
    return results


# ============================================================
# CONFIGURATION
# ============================================================

def get_config(attention_type, dataset_path=None):
    config = edict()
    
    config.network = "r100"
    config.embedding_size = 512
    config.fp16 = True
    
    config.attention_type = attention_type
    config.attention_reduction = 64
    
    config.scale = 64.0
    config.m1 = 0.5
    config.m2 = 0.1
    
    config.optimizer = "sgd"
    config.lr = 0.1
    config.batch_size = 256
    config.num_epoch = 25
    config.warmup_epoch = 1
    config.weight_decay = 5e-4
    config.momentum = 0.9
    config.gradient_acc = 1
    
    config.rec = dataset_path or "/slurm/homes/bel/Atten dualFace Project /faces_emore/faces_emore"
    config.num_classes = 85742
    config.num_image = 5822653
    
    config.val_targets = ["lfw", "cfp_fp", "agedb_30", "calfw","cfp_ff","cplfw"]
    
    config.output = f"output/ablation_{attention_type}"
    config.save_all_states = True
    config.resume = False
    config.verbose = 2500
    config.frequent = 100
    
    config.seed = 2048
    config.num_workers = 8
    config.dali = False
    config.dali_aug = False
    config.sample_rate = 1.0
    
    return config


ABLATION_INFO = {
    'dual_attention': ('★ BASELINE', 'Original simple MLP attention'),
    'uncertainty': ('◆ ABLATION', 'Log-variance uncertainty estimation'),
    'channel':  ('◆ ABLATION', 'Squeeze-Excitation feature refinement'),
    'hybrid': ('◆ FULL MODEL', 'Channel + Uncertainty combined'),
}


# ============================================================
# TRAINING
# ============================================================

def setup_distributed():
    try:
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        distributed.init_process_group("nccl")
    except KeyError:
        rank = 0
        local_rank = 0
        world_size = 1
        distributed.init_process_group(
            backend="nccl",
            init_method="tcp://127.0.0.1:12584",
            rank=rank,
            world_size=world_size,
        )
    return rank, local_rank, world_size


def setup_logging(output_dir, rank):
    os.makedirs(output_dir, exist_ok=True)
    log_file = os.path.join(output_dir, "training.log")
    
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ] if rank == 0 else [logging.FileHandler(log_file)]
    )


def setup_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class AverageMeter: 
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def train(cfg, rank, local_rank, world_size):
    """Main training function"""
    
    setup_seed(cfg.seed)
    torch.cuda.set_device(local_rank)
    setup_logging(cfg.output, rank)
    
    marker, desc = ABLATION_INFO.get(cfg.attention_type, ('? ', 'Unknown'))
    
    logging.info("=" * 70)
    logging.info(f"{marker} {cfg.attention_type.upper()}")
    logging.info(f"   {desc}")
    logging.info("=" * 70)
    
    # Import here to avoid TensorBoard issues
    from backbones import get_model
    from dataset import get_dataloader
    from lr_scheduler import PolynomialLRWarmup
    
    # Data
    train_loader = get_dataloader(
        cfg.rec, local_rank, cfg.batch_size, 
        cfg.dali, cfg.dali_aug, cfg.seed, cfg.num_workers
    )
    
    # Model
    backbone = get_model(cfg.network, dropout=0.0, fp16=cfg.fp16, num_features=cfg.embedding_size).cuda()
    backbone = torch.nn.parallel.DistributedDataParallel(
        backbone, broadcast_buffers=False, device_ids=[local_rank],
        bucket_cap_mb=16, find_unused_parameters=True
    )
    backbone.register_comm_hook(None, fp16_compress_hook)
    backbone.train()
    backbone._set_static_graph()
    
    logging.info(f"Backbone: {cfg.network}, Params: {sum(p.numel() for p in backbone.parameters()):,}")
    
    # Loss
    module_fc = AttenDualPartialFC(
        embedding_size=cfg.embedding_size,
        num_classes=cfg.num_classes,
        scale=cfg.scale,
        m1=cfg.m1,
        m2=cfg.m2,
        attention_type=cfg.attention_type,
        reduction=cfg.attention_reduction,
    ).cuda()
    module_fc.train()
    
    # Optimizer
    opt = torch.optim.SGD(
        [{"params": backbone.parameters()}, {"params": module_fc.parameters()}],
        lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay
    )
    
    # Scheduler
    cfg.total_batch_size = cfg.batch_size * world_size
    cfg.warmup_step = cfg.num_image // cfg.total_batch_size * cfg.warmup_epoch
    cfg.total_step = cfg.num_image // cfg.total_batch_size * cfg.num_epoch
    
    lr_scheduler = PolynomialLRWarmup(opt, cfg.warmup_step, cfg.total_step)
    
    # Resume
    start_epoch = 0
    global_step = 0
    if cfg.resume:
        ckpt = os.path.join(cfg.output, f"checkpoint_gpu_{rank}.pt")
        if os.path.exists(ckpt):
            checkpoint = torch.load(ckpt)
            start_epoch = checkpoint["epoch"]
            global_step = checkpoint["global_step"]
            backbone.module.load_state_dict(checkpoint["state_dict_backbone"])
            module_fc.load_state_dict(checkpoint["state_dict_fc"])
            opt.load_state_dict(checkpoint["state_optimizer"])
            lr_scheduler.load_state_dict(checkpoint["state_lr_scheduler"])
            logging.info(f"Resumed from epoch {start_epoch}")
    
    # Training
    loss_am = AverageMeter()
    atten_am = AverageMeter()
    amp = torch.cuda.amp.GradScaler(growth_interval=100)
    
    results = {'attention_type': cfg.attention_type, 'training':  [], 'validation': {}}
    
    logging.info(f"Training:  {cfg.num_epoch} epochs, {cfg.total_step} steps")
    
    for epoch in range(start_epoch, cfg.num_epoch):
        if isinstance(train_loader, DataLoader):
            train_loader.sampler.set_epoch(epoch)
        
        for img, labels in train_loader:
            global_step += 1
            
            embeddings = backbone(img)
            loss = module_fc(embeddings, labels)
            
            if cfg.fp16:
                amp.scale(loss).backward()
                if global_step % cfg.gradient_acc == 0:
                    amp.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(backbone.parameters(), 5)
                    amp.step(opt)
                    amp.update()
                    opt.zero_grad()
            else:
                loss.backward()
                if global_step % cfg.gradient_acc == 0:
                    torch.nn.utils.clip_grad_norm_(backbone.parameters(), 5)
                    opt.step()
                    opt.zero_grad()
            
            lr_scheduler.step()
            
            loss_am.update(loss.item(), 1)
            atten_am.update(module_fc.get_attention_score(), 1)
            
            if global_step % cfg.frequent == 0:
                lr = lr_scheduler.get_last_lr()[0]
                logging.info(
                    f"{marker} [{cfg.attention_type}] "
                    f"E[{epoch}/{cfg.num_epoch}] S[{global_step}/{cfg.total_step}] "
                    f"Loss:{loss_am.avg:.4f} Attn:{atten_am.avg:.3f} LR:{lr:.6f}"
                )
            
            if global_step % cfg.verbose == 0 and global_step > 0 and rank == 0:
                logging.info("Running validation...")
                val_results = validate(backbone.module, cfg.rec, cfg.val_targets)
                results['validation'][global_step] = val_results
        
        # Epoch end
        results['training'].append({
            'epoch':  epoch, 
            'loss': loss_am.avg, 
            'attention': atten_am.avg
        })
        
        logging.info(f"Epoch {epoch} done | Loss: {loss_am.avg:.4f} | Attn: {atten_am.avg:.3f}")
        
        # Save
        if cfg.save_all_states: 
            torch.save({
                "epoch": epoch + 1,
                "global_step":  global_step,
                "state_dict_backbone": backbone.module.state_dict(),
                "state_dict_fc":  module_fc.state_dict(),
                "state_optimizer":  opt.state_dict(),
                "state_lr_scheduler":  lr_scheduler.state_dict()
            }, os.path.join(cfg.output, f"checkpoint_gpu_{rank}.pt"))
        
        if rank == 0:
            torch.save(backbone.module.state_dict(), os.path.join(cfg.output, "model.pt"))
        
        if cfg.dali:
            train_loader.reset()
    
    # Final
    if rank == 0:
        torch.save(backbone.module.state_dict(), os.path.join(cfg.output, "model.pt"))
        with open(os.path.join(cfg.output, "results.json"), 'w') as f:
            json.dump(results, f, indent=2)
        logging.info(f"Done! Model:  {cfg.output}/model.pt")
    
    return results


def main():
    parser = argparse.ArgumentParser(description="Ablation Study Training")
    parser.add_argument("--attention", type=str, default="dual_attention",
                       choices=['dual_attention', 'uncertainty', 'channel', 'hybrid'])
    parser.add_argument("--run-all", action="store_true")
    parser.add_argument("--list", action="store_true")
    parser.add_argument("--dataset", type=str, default=None)
    parser.add_argument("--resume", action="store_true")

    # NumPy fix
    np.bool = np.bool_
    np.int = np.int_
    np.float = np.float64
    np.complex = np.complex128
    np.object = np.object_
    np.str = np.str_
    
    args = parser.parse_args()
    
    if args.list:
        print("\Available configurations:")
        print("=" * 50)
        for name, (marker, desc) in ABLATION_INFO.items():
            print(f"  {marker} {name}:  {desc}")
        print("=" * 50)
        return
    
    rank, local_rank, world_size = setup_distributed()
    torch.backends.cudnn.benchmark = True
    
    if args.run_all:
        all_results = {}
        for att_type in ['dual_attention', 'uncertainty', 'channel', 'hybrid']:
            print(f"\{'='*70}")
            print(f"Training:  {att_type}")
            print(f"{'='*70}\")
            cfg = get_config(att_type, args.dataset)
            all_results[att_type] = train(cfg, rank, local_rank, world_size)
        
        if rank == 0:
            with open("/slurm/homes/bel/Atten dualFace Project /output/ablation_all_results.json", 'w') as f:
                json.dump(all_results, f, indent=2)
            print("\✓ All ablations complete!")
    else:
        cfg = get_config(args.attention, args.dataset)
        if args.resume:
            cfg.resume = True
        train(cfg, rank, local_rank, world_size)


if __name__ == "__main__":
    main()
'''

with open('train_ablation.py', 'w') as f:
    f.write(ablation_train)

print("✓ Created self-contained train_ablation.py")

/slurm/homes/bel/Atten dualFace Project /insightface/recognition/arcface_torch
✓ Created self-contained train_ablation.py


In [12]:
!pwd


/slurm/homes/bel/Atten dualFace Project /insightface/recognition/arcface_torch


In [13]:
!ls -la *.py | grep -E "(attention|ablation|losses_attention)"
!ls -la configs/ablation*

-rw-r--r-- 1 bel user  5167 Dec 16 20:50 attention_modules.py
-rw-r--r-- 1 bel user  4850 Dec 16 20:50 losses_attention.py
-rw-r--r-- 1 bel user 20299 Dec 16 20:50 train_ablation.py
-rw-r--r-- 1 bel user 2584 Dec 16 20:50 configs/ablation_config.py


In [14]:
!pip install numpy==1.25.2 -q

In [15]:
!pip install sympy

Defaulting to user installation because normal site-packages is not writeable


In [17]:
%cd "/slurm/homes/bel/Atten dualFace Project /insightface/recognition/arcface_torch"

# Single GPU training
# !python train_kaggle.py configs/ms1mv2_kaggle

#!python train_ablation.py --attention hybrid
!python train_ablation.py --attention hybrid
!python train_ablation_tb_El_OanasNewATT.py --attention dual_attention

/slurm/homes/bel/Atten dualFace Project /insightface/recognition/arcface_torch
2025-12-16 20:50:44,280 ======================================================================
2025-12-16 20:50:44,280 ◆ FULL MODEL HYBRID
2025-12-16 20:50:44,280    Channel + Uncertainty combined
2025-12-16 20:50:44,280 ======================================================================
/slurm/homes/bel/.local/lib/python3.11/site-packages/torch/utils/_contextlib.py:125: UserWarning: Decorating classes is deprecated and will be disabled in future versions. You should only decorate functions or methods. To preserve the current behavior of class decoration, you can directly decorate the `__init__` method and nothing else.
  warnings.warn("Decorating classes is deprecated and will be disabled in "
/slurm/homes/bel/.local/lib/python3.11/site-packages/torch/nn/parallel/distributed.py:2219: UserWarning: You passed find_unused_parameters=true to DistributedDataParallel, `_set_static_graph` will detect unused par